# RAG desde cero con la Constitución de Chile

**RAG** (*Retrieval-Augmented Generation*) = antes de que el modelo responda, **buscamos** los fragmentos de texto relevantes y se los **pasamos en el prompt**. Así el modelo responde con información que no necesariamente conoce, y podemos citar la fuente.

Todo corre local con [Ollama](https://ollama.com), sin frameworks:

| Paso | Qué hace | Herramienta |
|---|---|---|
| 1. Chunks | Dividir el documento en trozos | `preparar.py` (un trozo por artículo) |
| 2. Embeddings | Convertir cada trozo en un vector que representa su significado | `bge-m3` |
| 3. Búsqueda | Encontrar los trozos más parecidos a la pregunta | similitud coseno con NumPy |
| 4. Generación | Responder usando solo esos trozos | `qwen3:4b-instruct` |

Requisitos: Ollama corriendo y los modelos descargados (`ollama pull bge-m3` y `ollama pull qwen3:4b-instruct`).

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import ollama

MODELO_EMBEDDINGS = "bge-m3"
MODELO_LLM = "qwen3:4b-instruct"

## 1. Chunks

`preparar.py` limpió el texto de la BCN (quitó las notas al margen sobre qué ley modificó cada inciso) y lo dividió **por artículo**. Los artículos largos, como el 19, se partieron en trozos de ~1.500 caracteres.

Cortar por la estructura natural del documento suele funcionar mejor que cortar cada N caracteres: cada trozo es una idea completa y trae su propia "dirección" (artículo y capítulo) para citarla.

In [2]:
chunks = [json.loads(l) for l in Path("data/chunks.jsonl").read_text(encoding="utf-8").splitlines()]
largos = [len(c["texto"]) for c in chunks]
print(f"{len(chunks)} chunks · largo promedio {sum(largos) // len(largos)} caracteres · máximo {max(largos)}")
chunks[0]

378 chunks · largo promedio 954 caracteres · máximo 2060


{'id': 0,
 'fuente': 'Artículo 1 (Capítulo I)',
 'texto': 'Artículo 1°.- Las personas nacen libres e iguales en dignidad y derechos.\nLa familia es el núcleo fundamental de la sociedad.\nEl Estado reconoce y ampara a los grupos intermedios a través de los cuales se organiza y estructura la sociedad y les garantiza la adecuada autonomía para cumplir sus propios fines específicos.\nEl Estado está al servicio de la persona humana y su finalidad es promover el bien común, para lo cual debe contribuir a crear las condiciones sociales que permitan a todos y a cada uno de los integrantes de la comunidad nacional su mayor realización espiritual y material posible, con pleno respeto a los derechos y garantías que esta Constitución establece.\nEs deber del Estado resguardar la seguridad nacional, dar protección a la población y a la familia, propender al fortalecimiento de ésta, promover la integración armónica de todos los sectores de la Nación y asegurar el derecho de las personas a participar

## 2. Embeddings

Un **embedding** es una lista de números (un vector) que representa el significado de un texto. Textos con significado parecido quedan cerca entre sí, aunque usen palabras distintas.

Usamos `bge-m3` porque es multilingüe y funciona bien en español. Antes de calcular el embedding, le agregamos al texto su fuente: así "Artículo 19" también es parte de lo que se busca. Se calculan una sola vez y se guardan en disco.

In [3]:
def embeber(textos: list[str]) -> np.ndarray:
    """Devuelve un vector normalizado (largo 1) por texto."""
    vectores = np.array(ollama.embed(model=MODELO_EMBEDDINGS, input=textos)["embeddings"], dtype=np.float32)
    return vectores / np.linalg.norm(vectores, axis=1, keepdims=True)


CACHE = Path("data/embeddings.npy")
if CACHE.exists():
    matriz = np.load(CACHE)
else:
    inicio = time.time()
    textos = [f"{c['fuente']}\n{c['texto']}" for c in chunks]
    matriz = np.vstack([embeber(textos[i : i + 32]) for i in range(0, len(textos), 32)])
    np.save(CACHE, matriz)
    print(f"Embeddings calculados en {time.time() - inicio:.0f} s")

print("Forma de la matriz:", matriz.shape, "→ un vector de", matriz.shape[1], "dimensiones por chunk")
print("Primeros valores del chunk 0:", matriz[0][:6].round(3))

Forma de la matriz: (378, 1024) → un vector de 1024 dimensiones por chunk
Primeros valores del chunk 0: [ 0.013  0.019 -0.046 -0.022 -0.004 -0.04 ]


**¿Capturan significado?** Comparemos frases con la *similitud coseno* (1 = mismo significado, 0 = nada que ver). Como los vectores están normalizados, es simplemente el producto punto.

In [4]:
frases = [
    "Las personas nacen libres e iguales en dignidad y derechos.",
    "Todos los seres humanos tienen la misma dignidad desde que nacen.",
    "El Presidente de la República dura cuatro años en su cargo.",
]
v = embeber(frases)
for i in range(1, 3):
    print(f"{v[0] @ v[i]:.3f}  ← «{frases[0][:40]}…» vs «{frases[i][:40]}…»")

0.850  ← «Las personas nacen libres e iguales en d…» vs «Todos los seres humanos tienen la misma …»
0.397  ← «Las personas nacen libres e iguales en d…» vs «El Presidente de la República dura cuatr…»


## 3. Búsqueda (la "R" de RAG)

Para buscar: convertimos la pregunta en un vector y lo comparamos contra **todos** los chunks. Con 378 chunks basta un producto de matrices; con millones se usaría una base de datos vectorial (Chroma, pgvector, Qdrant…), que hace lo mismo pero con índices aproximados.

In [5]:
def buscar(pregunta: str, k: int = 4) -> list[tuple[float, dict]]:
    q = embeber([pregunta])[0]
    puntajes = matriz @ q
    mejores = np.argsort(-puntajes)[:k]
    return [(float(puntajes[i]), chunks[i]) for i in mejores]


for puntaje, c in buscar("¿Cuánto dura el mandato del Presidente?"):
    print(f"{puntaje:.3f}  {c['fuente']}\n       {c['texto'][:150]}…\n")

0.683  Artículo 25 (Capítulo IV)
       Artículo 25.- Para ser elegido Presidente de la República se requiere tener la nacionalidad chilena de acuerdo a lo dispuesto en los números 1º ó 2º d…

0.633  Artículo 29 (Capítulo IV), parte 2 de 2
       Si la vacancia se produjere faltando dos años o más para la próxima elección presidencial, el Vicepresidente, dentro de los diez primeros días de su m…

0.613  Artículo 27 (Capítulo IV)
       Artículo 27.- El proceso de calificación de la elección presidencial deberá quedar concluido dentro de los quince días siguientes tratándose de la pri…

0.606  Artículo 26 (Capítulo IV), parte 1 de 2
       Artículo 26.- El Presidente de la República será elegido en votación directa y por mayoría absoluta de los sufragios válidamente emitidos. La elección…



## 4. Generación (la "G" de RAG)

Armamos un prompt con los chunks encontrados, numerados, y le pedimos al modelo que responda **solo** con esa información y cite la fuente. Si la respuesta no está en el contexto, debe decirlo en vez de inventar.

In [6]:
INSTRUCCIONES = """Eres un asistente que responde preguntas sobre la Constitución Política de Chile.
Responde en español, en forma breve, usando SOLO la información del contexto.
Cita entre corchetes la fuente de cada afirmación, por ejemplo [Artículo 25 (Capítulo IV)].
Si el contexto no contiene la respuesta, di exactamente: "No encontré eso en la Constitución.\""""


def preguntar_llm(mensaje: str) -> str:
    r = ollama.chat(model=MODELO_LLM, messages=[{"role": "user", "content": mensaje}], options={"temperature": 0})
    return r["message"]["content"].strip()


def rag(pregunta: str, k: int = 4, mostrar_fuentes: bool = True) -> str:
    encontrados = buscar(pregunta, k)
    contexto = "\n\n".join(f"[{c['fuente']}]\n{c['texto']}" for _, c in encontrados)
    respuesta = preguntar_llm(f"{INSTRUCCIONES}\n\n### Contexto\n{contexto}\n\n### Pregunta\n{pregunta}")
    if mostrar_fuentes:
        print("Fuentes recuperadas:", *[f"  {p:.3f} {c['fuente']}" for p, c in encontrados], sep="\n")
    return respuesta


print(rag("¿Cuánto dura el mandato del Presidente y puede ser reelegido?"))

Fuentes recuperadas:
  0.681 Artículo 25 (Capítulo IV)
  0.620 Artículo 26 (Capítulo IV), parte 1 de 2
  0.617 Artículo 29 (Capítulo IV), parte 2 de 2
  0.595 Artículo 27 (Capítulo IV)
El mandato del Presidente dura cuatro años y no puede ser reelegido para el período siguiente [Artículo 25 (Capítulo IV)].


## 5. ¿Sirvió de algo? Con RAG vs. sin RAG

Misma pregunta, mismo modelo. Sin RAG el modelo responde con lo que "recuerda" de su entrenamiento: un modelo chico suele equivocarse en detalles o no conocer reformas recientes.

In [7]:
preguntas = [
    "¿A qué edad se puede votar en Chile y es obligatorio votar?",
    "¿Cuántos senadores tiene el Senado?",
    "¿Qué requisitos hay para ser elegido diputado?",
]
for p in preguntas:
    print("❓", p)
    print("\n— Sin RAG —\n" + preguntar_llm(p + " Responde brevemente, según la Constitución de Chile."))
    print("\n— Con RAG —\n" + rag(p, mostrar_fuentes=False))
    print("\n" + "=" * 80 + "\n")

❓ ¿A qué edad se puede votar en Chile y es obligatorio votar?



— Sin RAG —
En Chile, se puede votar a partir de los **18 años** y **es obligatorio votar**.  

Según la **Constitución de Chile (1980)**, el derecho a votar se otorga a todos los ciudadanos mayores de **18 años**, y el voto es **obligatorio** para todos los ciudadanos mayores de 18 años que estén inscritos en el padrón electoral.  

Artículo 43 de la Constitución: *"El voto es obligatorio para todos los ciudadanos mayores de dieciocho años."*  

✅ Edad mínima: 18 años  
✅ Obligatoriedad: Sí, es obligatorio para los mayores de 18 años inscritos.



— Con RAG —
Se puede votar en Chile a los dieciocho años de edad [Artículo 13 (Capítulo II)].  
El voto es obligatorio para quienes tengan domicilio electoral en Chile [Artículo 160 (Capítulo XV)].


❓ ¿Cuántos senadores tiene el Senado?



— Sin RAG —
El Senado de Chile está compuesto por **38 senadores**, según la Constitución de Chile (2022). Esta cantidad se establece en el artículo 133, que distribuye los 38 senadores entre 14 regiones, con 2 senadores por región, y 2 senadores adicionales por región en el caso de las regiones más grandes (como la región de Valparaíso y la de Santiago), aunque la distribución exacta se ajusta a la ley electoral.  

Sin embargo, en la Constitución de Chile, el número total de senadores es **38**, y se eligen por distritos regionales.  

**Respuesta breve:**  
El Senado de Chile tiene **38 senadores**.



— Con RAG —
No encontré eso en la Constitución.


❓ ¿Qué requisitos hay para ser elegido diputado?



— Sin RAG —
Según la Constitución de Chile, para ser elegido diputado se requiere:

- Ser chileno por nacimiento o por nacionalidad.  
- Tener al menos 25 años de edad.  
- Ser mayor de 25 años y tener residencia en el país.  
- No estar inhabilitado por una ley o por un juicio de inmoralidad.  

Además, el artículo 130 de la Constitución establece que los diputados se eligen por voto popular en distritos electorales, y que el número de diputados es fijo, con un límite de 120.

(Nota: La Constitución de Chile de 2022 establece que los diputados son elegidos por un sistema de representación proporcional, con un mínimo de 120 diputados, y que los requisitos de edad, nacionalidad y residencia se mantienen como condiciones previas para su candidatura.)



— Con RAG —
Para ser elegido diputado se requiere ser ciudadano con derecho a sufragio, tener cumplidos veintiún años de edad, haber cursado la enseñanza media o equivalente, y tener residencia en la región a que pertenezca el distrito electoral correspondiente durante un plazo no inferior a dos años, contado hacia atrás desde el día de la elección [Artículo 48 (Capítulo V)].




## 6. Límites

Dos pruebas para ver dónde falla o cómo se comporta:
1. Una pregunta **fuera del documento**: debería decir que no lo encontró.
2. Una pregunta **con otras palabras** que las del texto: la búsqueda semántica debería encontrarla igual.

In [8]:
print(rag("¿Quién ganó la Copa América 2015?"))
print()
print(rag("¿Me pueden meter preso sin una orden?"))

Fuentes recuperadas:
  0.338 Artículo 154 (Capítulo XV), parte 1 de 3
  0.322 Artículo 15 (Capítulo II)
  0.321 Artículo 154 (Capítulo XV), parte 2 de 3
  0.318 Artículo 5 (Capítulo I)
No encontré eso en la Constitución.



Fuentes recuperadas:
  0.601 Artículo 19 (Capítulo III), parte 5 de 21
  0.569 Artículo 21 (Capítulo III)
  0.551 Artículo 19 (Capítulo III), parte 4 de 21
  0.537 Artículo 81 (Capítulo VI)
No, no pueden meter preso sin una orden [Artículo 19 (Capítulo III), parte 4 de 21].


## Conclusiones

- Un RAG funcional son ~40 líneas: **embeber**, **comparar**, **meter en el prompt**.
- La calidad depende sobre todo de los **chunks**: limpiar las notas al margen y cortar por artículo fue la parte que más trabajo dio.
- Citar la fuente sale casi gratis, porque cada chunk ya la trae.
- Siguientes pasos posibles: medir la búsqueda con un set de preguntas y respuestas esperadas, probar búsqueda híbrida (palabras clave + embeddings) y un *re-ranker*.